# Human-in-the-Loop with Pydantic AI

Human-in-the-loop (HITL) workflows allow a human to review and approve agent actions before they are executed. This is critical for high-stakes operations.

Pydantic AI supports this natively through **deferred tools**: tools that require approval before execution.

```mermaid
sequenceDiagram
    participant User
    participant Agent
    participant LLM

    User->>Agent: Question
    Agent->>LLM: Prompt + Tool definitions
    LLM->>Agent: Tool call request
    Agent->>User: DeferredToolRequests (approval needed)
    User->>Agent: DeferredToolResults (approved/denied)
    Agent->>LLM: Tool results or denial feedback
    LLM->>Agent: Final response
    Agent->>User: Output
```

Key concepts:
- **`requires_approval=True`**: Mark tools that always need human approval
- **`ApprovalRequired`**: Raise conditionally when approval depends on arguments or context
- **`ctx.tool_call_approved`**: Check if a tool call has already been approved
- **`DeferredToolRequests`**: Returned when tools need approval (contains pending approvals)
- **`DeferredToolResults`**: Pass back approval decisions to resume the agent

Reference: https://ai.pydantic.dev/deferred-tools/#human-in-the-loop-tool-approval

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
from dotenv import load_dotenv
from pydantic_ai import (
    Agent,
    ApprovalRequired,
    DeferredToolRequests,
    DeferredToolResults,
    RunContext,
    ToolDenied,
)

load_dotenv()

## Always require approval

Use `requires_approval=True` on the tool decorator. When the LLM calls this tool, the agent returns a `DeferredToolRequests` instead of executing it.

The output type must be `[str, DeferredToolRequests]` so the agent can return either a final answer or pending approvals.

In [ ]:
agent = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant that can run python code.",
    output_type=[str, DeferredToolRequests],
)


# THIS IS DANGEROUS, DO NOT USE IN PRODUCTION
@agent.tool_plain(requires_approval=True)
def run_python_code(code: str) -> str:
    """Run arbitrary Python code. Do not use any external libraries.

    Args:
        code: Python code to run
    """
    import sys
    from io import StringIO

    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    namespace = {}

    try:
        exec(code, namespace)
        output = captured_output.getvalue()
        if not output.strip():
            user_vars = {
                k: v
                for k, v in namespace.items()
                if not k.startswith("__") and k not in ["StringIO", "sys"]
            }
            if user_vars:
                output = str(
                    user_vars if len(user_vars) > 1 else list(user_vars.values())[0]
                )
        return output.strip() if output.strip() else "Code executed successfully"
    except Exception as e:
        return f"Error: {str(e)}"
    finally:
        sys.stdout = old_stdout


# THIS IS DANGEROUS, DO NOT USE IN PRODUCTION

### Step 1: Run the agent

The agent will return `DeferredToolRequests` with pending tool calls that need approval.

In [ ]:
result = agent.run_sync("Give me 5 random numbers")
messages = result.all_messages()

assert isinstance(result.output, DeferredToolRequests)
requests = result.output

print(f"Pending approvals: {len(requests.approvals)}")
for call in requests.approvals:
    print(f"  Tool: {call.tool_name}")
    print(f"  Args: {call.args}")
    print(f"  ID:   {call.tool_call_id}")

### Step 2: Approve and resume

Create `DeferredToolResults` with approval decisions, then resume the agent with the original message history.

In [ ]:
# Approve all pending tool calls
results = DeferredToolResults()
for call in requests.approvals:
    results.approvals[call.tool_call_id] = True

# Resume the agent
result = agent.run_sync(
    message_history=messages,
    deferred_tool_results=results,
)
print(result.output)

### Denying a tool call

Use `ToolDenied` to reject a tool call with a reason. The agent will receive the denial as feedback and try a different approach.

In [ ]:
result = agent.run_sync("Delete all files in the current directory using Python")
messages = result.all_messages()

assert isinstance(result.output, DeferredToolRequests)
requests = result.output

# Deny the tool call
results = DeferredToolResults()
for call in requests.approvals:
    results.approvals[call.tool_call_id] = ToolDenied(
        "Deleting files is not allowed. Just list the files instead."
    )

result = agent.run_sync(
    message_history=messages,
    deferred_tool_results=results,
)
print(result.output)

## Conditional approval with `ApprovalRequired`

Sometimes approval depends on the tool arguments. Use `ApprovalRequired` to conditionally require approval, and `ctx.tool_call_approved` to check if the call was already approved.

```mermaid
flowchart LR
    LLM -->|"Tool call"| Check{"Protected?"}
    Check -->|"No"| Execute["Execute directly"]
    Check -->|"Yes"| Approved{"Approved?"}
    Approved -->|"Yes"| Execute
    Approved -->|"No"| Defer["DeferredToolRequests"]
    Defer -->|"Human review"| Resume["Resume agent"]
```

In [ ]:
file_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You are a helpful assistant that can read and update files. "
        "Use the provided tools to perform file operations."
    ),
    output_type=[str, DeferredToolRequests],
)

PROTECTED_FILES = {".env", "secrets.yaml", "credentials.json"}


@file_agent.tool_plain
def read_file(path: str) -> str:
    """Read the contents of a file.

    Args:
        path: Path to the file to read.
    """
    # Simulate reading a file
    return f"Contents of {path!r}: [simulated file content]"


@file_agent.tool
def update_file(ctx: RunContext, path: str, content: str) -> str:
    """Write content to a file. Requires approval for protected files.

    Args:
        ctx: The call context.
        path: Path to the file to write.
        content: Content to write to the file.
    """
    if path in PROTECTED_FILES and not ctx.tool_call_approved:
        raise ApprovalRequired(metadata={"reason": "protected file"})
    return f"File {path!r} updated with: {content!r}"


@file_agent.tool_plain(requires_approval=True)
def delete_file(path: str) -> str:
    """Delete a file. Always requires approval.

    Args:
        path: Path to the file to delete.
    """
    return f"File {path!r} deleted"

### Non-protected file: no approval needed

In [ ]:
result = file_agent.run_sync("Write 'Hello, world!' to README.md")
print(type(result.output).__name__, "-", result.output)

### Protected file: approval required

In [ ]:
result = file_agent.run_sync(
    "Write 'API_KEY=abc123' to .env and delete old_config.yaml"
)
messages = result.all_messages()

assert isinstance(result.output, DeferredToolRequests)
requests = result.output

print(f"Pending approvals: {len(requests.approvals)}")
for call in requests.approvals:
    print(f"  Tool: {call.tool_name}, Args: {call.args}")
    if call.tool_call_id in requests.metadata:
        print(f"  Metadata: {requests.metadata[call.tool_call_id]}")

In [ ]:
# Approve the .env update but deny the delete
results = DeferredToolResults()
for call in requests.approvals:
    if call.tool_name == "update_file":
        results.approvals[call.tool_call_id] = True
    elif call.tool_name == "delete_file":
        results.approvals[call.tool_call_id] = ToolDenied(
            "Deleting config files is not allowed."
        )

result = file_agent.run_sync(
    message_history=messages,
    deferred_tool_results=results,
)
print(result.output)

## Interactive approval loop

You can build a reusable approval loop that prompts the user for each pending tool call.

In [ ]:
def run_with_approval(agent: Agent, prompt: str):
    """Run an agent with interactive human approval for deferred tools."""
    result = agent.run_sync(prompt)

    while isinstance(result.output, DeferredToolRequests):
        messages = result.all_messages()
        requests = result.output

        results = DeferredToolResults()
        for call in requests.approvals:
            print(f"\n--- Approval required ---")
            print(f"Tool: {call.tool_name}")
            print(f"Args: {call.args}")
            answer = input("Approve? (yes/no): ").strip().lower()

            if answer == "yes":
                results.approvals[call.tool_call_id] = True
            else:
                reason = input("Reason for denial: ").strip()
                results.approvals[call.tool_call_id] = ToolDenied(
                    reason or "User denied this action."
                )

        result = agent.run_sync(
            message_history=messages,
            deferred_tool_results=results,
        )

    return result.output


output = run_with_approval(agent, "Give me 10 random numbers")
print(f"\nFinal output: {output}")

## Exercise

Build an agent with tools for managing a bank account (check balance, transfer money, close account). Use conditional approval:
- `check_balance`: no approval needed
- `transfer_money`: approval required only for amounts > $1000
- `close_account`: always requires approval